# Error by Geography - Median Error Run

This notebook extends notebook 4 (Error by Geography) to use the **median-error run** from the
retraining stability analysis (notebook 9b).

**Rationale:**
- Instead of using a single arbitrary model or averaging across all 10 runs,
  we select the run with the median overall RMSE.
- This gives the most representative/typical result while avoiding outliers.

**Approach:**
1. Load reconstruction errors from all 10 retraining runs
2. Identify the run with the median overall RMSE
3. Use that run's errors for geographic analysis (matching notebook 4 style)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import pickle
from textwrap import fill

# Set style
sns.set(style="white")

# Configuration
bottleneck = 100  # Focus on 100D (matches original notebook 4)
n_runs = 10

print(f"Analyzing reconstruction error by geography using median-error run")
print(f"Bottleneck dimension: {bottleneck}")
print(f"Number of runs to select from: {n_runs}")

## 1. Load Auxiliary Geographic Data

In [ ]:
# Load the lookup tables
oa_lsoa = pd.read_csv('../data/geofiles/lookup_oa2022_lsoa11_EW.csv')
oa_lsoa.set_index('OA21CD', inplace=True)

oa_msoa = pd.read_csv('../data/geofiles/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv')[["OA21CD","MSOA21CD"]].set_index('OA21CD')

# Load the IMD data
imd = pd.read_csv('../data/geofiles/uk_imd2019.csv')
imd = imd[["LSOA","SOA_decile"]]
imd.columns = ['LSOA11CD','IMD']

# Load the population density data
density = pd.read_csv('../data/census_data/eng_raw_csvs/ts006.csv')
density.columns = ['OA21CD','Density']
density['Density_decile'] = pd.qcut(density['Density'], 10, labels=False)
density['Density_decile'] = 10 - density['Density_decile']  # reverse the order
density.drop('Density', axis=1, inplace=True)
density.set_index('OA21CD', inplace=True)

print(f"Loaded OA-LSOA lookup: {len(oa_lsoa)} rows")
print(f"Loaded OA-MSOA lookup: {len(oa_msoa)} rows")
print(f"Loaded IMD data: {len(imd)} rows")
print(f"Loaded density data: {len(density)} rows")

In [ ]:
# Load OAC data
OAC = pd.read_csv("../data/OAC/OAC_assignment.csv")
OAC = OAC[["Geography_Code", "Supergroup8", "Group", "Subgroup"]]
OAC = OAC.rename(columns={"Geography_Code": "OA21CD"})

# Load the OAC category names
OAC_cats = pd.read_csv("../data/OAC/OAC_cats.csv")
OAC_cats = OAC_cats[['Classification Code', 'Classification Name']]

# Make a dict out of the first 8 rows
OAC_cats_dict = OAC_cats.set_index('Classification Code')['Classification Name'].to_dict()

# Convert codes to string before mapping
OAC['Supergroup8'] = OAC['Supergroup8'].astype(str)
OAC['Supergroup_name'] = OAC['Supergroup8'].map(OAC_cats_dict)
OAC['Supergroup_codename'] = OAC['Supergroup8'] + " - " + OAC['Supergroup_name']
OAC['Group'] = OAC['Group'].astype(str)
OAC['Group_name'] = OAC['Group'].map(OAC_cats_dict)
OAC['Subgroup'] = OAC['Subgroup'].astype(str)
OAC['Subgroup_name'] = OAC['Subgroup'].map(OAC_cats_dict)

print(f"Loaded OAC data: {len(OAC)} rows")
print(f"Unique supergroups: {OAC['Supergroup_codename'].nunique()}")

## 2. Load Stability Results and Identify Median-Error Run

In [ ]:
# Load the stability results
stability_path = f"../AE_outputs/retraining_stability/data/stability_checkpoint_{bottleneck}d.pkl"

with open(stability_path, 'rb') as f:
    checkpoint = pickle.load(f)

print(f"Loaded checkpoint for {bottleneck}D")
print(f"Keys: {checkpoint.keys()}")
print(f"Number of runs: {checkpoint['n_runs']}")

# Get the reconstruction errors - these are MSE per OA
reco_errors_list = checkpoint['reco_errors_list']
print(f"Shape of reco errors for run 0: {reco_errors_list[0].shape}")

In [ ]:
# Compute overall RMSE for each run and find the median
rmse_per_run = []
for run_idx, reco_err in enumerate(reco_errors_list):
    # MSE to RMSE
    rmse = np.sqrt(reco_err.mean())
    rmse_per_run.append(rmse)
    print(f"Run {run_idx}: RMSE = {rmse*100:.4f}%")

rmse_per_run = np.array(rmse_per_run)

# Find the median run
# With 10 runs, median is between 5th and 6th sorted values
# We'll pick the run closest to the median value
median_rmse = np.median(rmse_per_run)
median_run_idx = np.argmin(np.abs(rmse_per_run - median_rmse))

print(f"\nOverall RMSE: {rmse_per_run.mean()*100:.4f}% ± {rmse_per_run.std()*100:.4f}%")
print(f"Median RMSE: {median_rmse*100:.4f}%")
print(f"\n>>> Selected median run: Run {median_run_idx} (RMSE = {rmse_per_run[median_run_idx]*100:.4f}%)")

In [ ]:
# Load census data to get OA identifiers
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
data = pd.read_parquet(cleaned_data_path)
data = data.reset_index()

oa_ids = data['OA'].values
print(f"Number of OAs: {len(oa_ids)}")

# Verify lengths match
assert len(oa_ids) == len(reco_errors_list[0]), "OA count mismatch!"

In [ ]:
# Load PCA reconstruction for comparison
pca_path = f"../data/AE_outputs/engcensus_all/PCA/{bottleneck}_components.csv"
pca_reco = pd.read_csv(pca_path, index_col=0)

# Compute PCA reconstruction error (RMSE per OA)
pca_err = np.sqrt(np.mean((data.set_index("OA") - pca_reco) ** 2, axis=1)).reset_index()
pca_err.columns = ['OA21CD', 'pca_err']
pca_err['pca_err'] = pca_err['pca_err'] * 100  # Convert to percentage

print(f"PCA reconstruction error computed")
print(f"Mean PCA RMSE: {pca_err['pca_err'].mean():.4f}%")

In [ ]:
# Create DataFrame using the MEDIAN RUN errors
median_reco_err = reco_errors_list[median_run_idx]

# Convert MSE to RMSE and multiply by 100 for percentage
reco_err = pd.DataFrame({
    'OA21CD': oa_ids,
    f'reco_err_{bottleneck}': np.sqrt(median_reco_err) * 100
})

# Normalize errors
reco_err[f'reco_err_{bottleneck}_norm'] = reco_err[f'reco_err_{bottleneck}'] / reco_err[f'reco_err_{bottleneck}'].mean() * 100

print(f"Created reconstruction error DataFrame using median run (Run {median_run_idx})")
print(f"Mean AE RMSE: {reco_err[f'reco_err_{bottleneck}'].mean():.4f}%")

In [ ]:
# Prepare PCA error DataFrame (matching format)
pca_reco_err = pca_err.copy()
pca_reco_err.columns = ['OA21CD', f'pca_err_{bottleneck}']
pca_reco_err[f'pca_err_{bottleneck}_norm'] = pca_reco_err[f'pca_err_{bottleneck}'] / pca_reco_err[f'pca_err_{bottleneck}'].mean() * 100

# Compute percentage difference
reco_err[f'perc_diff_{bottleneck}_ae'] = (reco_err[f'reco_err_{bottleneck}'] - pca_reco_err[f'pca_err_{bottleneck}']) / pca_reco_err[f'pca_err_{bottleneck}'] * 100

# Print comparison
n_ae_greater = len(reco_err[reco_err[f'reco_err_{bottleneck}'] > pca_reco_err[f'pca_err_{bottleneck}']])
print(f"Number of OAs where AE > PCA: {n_ae_greater} out of {len(reco_err)}")
print(f"Which is {n_ae_greater / len(reco_err) * 100:.2f}% of OAs")

In [ ]:
# Merge with geographic metadata
reco_err = reco_err.merge(OAC, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(OAC, on="OA21CD", how="left")

# Add density deciles
reco_err = reco_err.merge(density, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(density, on="OA21CD", how="left")

# Add LSOA
reco_err = reco_err.merge(oa_lsoa, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(oa_lsoa, on="OA21CD", how="left")

# Add MSOA
reco_err = reco_err.merge(oa_msoa, on="OA21CD", how="left")
pca_reco_err = pca_reco_err.merge(oa_msoa, on="OA21CD", how="left")

reco_err = reco_err.dropna()

# Add IMD
reco_err = reco_err.merge(imd, on="LSOA11CD", how="left").dropna()
pca_reco_err = pca_reco_err.merge(imd, on="LSOA11CD", how="left").dropna()

reco_err["IMD"] = reco_err["IMD"].astype(int)
pca_reco_err["IMD"] = pca_reco_err["IMD"].astype(int)

# Check length
if len(reco_err) != 188880 or len(pca_reco_err) != 188880:
    print(f"Warning: Length of reco_err is {len(reco_err)}")
    print(f"Warning: Length of pca_reco_err is {len(pca_reco_err)}")
else:
    print(f"Data merged successfully: {len(reco_err)} OAs")

## 3. Plotting Functions (from notebook 4)

In [ ]:
def prepare_grouped_data(df1, df2, group_col, bottleneck, norm=False):
    """Merge two datasets and compute mean reconstruction error for each group."""

    if norm:
        ae_col = f"reco_err_{bottleneck}_norm"
        pca_col = f"pca_err_{bottleneck}_norm"
    else:
        ae_col = f"reco_err_{bottleneck}"
        pca_col = f"pca_err_{bottleneck}"
    
    grouped_1 = df1.groupby(group_col)[ae_col].mean().reset_index()
    grouped_2 = df2.groupby(group_col)[pca_col].mean().reset_index()
    
    grouped = grouped_1.merge(grouped_2, on=group_col)
    grouped.rename(columns={ae_col: "AE", pca_col: "PCA"}, inplace=True)
    
    grouped_melted = grouped.melt(id_vars=[group_col], var_name="Error Type", value_name="Mean Reco Error")
    return grouped, grouped_melted


def plot_side_by_side_charts(df1, df2, group_col1, group_col2, xlabel1, xlabel2, title_prefix, palette, bottleneck, norm=False, rotation=45, wrapped_labels=False, print_means=False):
    """Creates a figure with two side-by-side subplots for IMD and Density Decile, with optional mean value printing."""
    
    grouped1, grouped_melted1 = prepare_grouped_data(df1, df2, group_col1, bottleneck, norm=norm)
    grouped2, grouped_melted2 = prepare_grouped_data(df1, df2, group_col2, bottleneck, norm=norm)
    grouped1["Error Difference"] = grouped1["AE"] - grouped1["PCA"]
    grouped1["Perc_Error_Change"] = (grouped1["AE"] - grouped1["PCA"]) / grouped1["PCA"] * 100
    grouped2["Error Difference"] = grouped2["AE"] - grouped2["PCA"]
    grouped2["Perc_Error_Change"] = (grouped2["AE"] - grouped2["PCA"]) / grouped2["PCA"] * 100

    if print_means:
        def print_summary(grouped_melted, group_col, label):
            means = grouped_melted.groupby([group_col, "Error Type"])["Mean Reco Error"].mean().unstack()
            print(f"\n=== Mean Reconstruction Error by Group ({label}) ===")
            print(means)

            if grouped_melted[group_col].dtype.name == "category":
                levels = grouped_melted[group_col].cat.categories
            else:
                levels = sorted(grouped_melted[group_col].unique())

            if len(levels) >= 2:
                first, last = levels[0], levels[-1]
                print(f"\n--- Percentage Increase in Error ({label}): {last} → {first} ---")
                for error_type in means.columns:
                    first_val = means.loc[first, error_type]
                    last_val = means.loc[last, error_type]
                    perc_diff = ((first_val - last_val) / last_val) * 100
                    print(f"{error_type}: {perc_diff:.2f}%")

        print_summary(grouped_melted1, group_col1, xlabel1)
        print_summary(grouped_melted2, group_col2, xlabel2)

    
    fig, axes = plt.subplots(2, 2, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]}, sharex='col')
    
    sns.barplot(ax=axes[0, 0], x=group_col1, y="Mean Reco Error", hue="Error Type", data=grouped_melted1, palette=palette)
    axes[0, 0].set_xlabel("")
    axes[0, 0].set_ylabel("% of Mean Reconstruction Error" if norm else "Mean Reconstruction Error (RMSE)")
    axes[0, 0].set_title(f"{title_prefix} by {xlabel1}")
    axes[0, 0].tick_params(axis='x', rotation=rotation)
    axes[0, 0].legend(title="Error Type", frameon=False)
    
    sns.barplot(ax=axes[0, 1], x=group_col2, y="Mean Reco Error", hue="Error Type", data=grouped_melted2, palette=palette)
    axes[0, 1].set_xlabel("")
    axes[0, 1].set_ylabel("")
    axes[0, 1].set_title(f"{title_prefix} by {xlabel2}")
    axes[0, 1].tick_params(axis='x', rotation=rotation)
    axes[0, 1].legend(title="Error Type", frameon=False)

    if norm:
        sns.barplot(ax=axes[1, 0], x=group_col1, y="Error Difference", data=grouped1, color="darkblue")
        axes[1, 0].axhline(0, color="red", linestyle="--")
        axes[1, 0].set_ylabel("Difference (AE - PCA) [%]")
    else:
        sns.barplot(ax=axes[1, 0], x=group_col1, y="Perc_Error_Change", data=grouped1, color="darkblue")
        axes[1, 0].axhline(grouped1["Perc_Error_Change"].mean(), color="red", linestyle="--")
        axes[1, 0].set_ylabel("(AE - PCA)/PCA [%]")
    axes[1, 0].set_xlabel(xlabel1)
    axes[1, 0].tick_params(axis='x', rotation=rotation)
    
    if norm:
        sns.barplot(ax=axes[1, 1], x=group_col2, y="Error Difference", data=grouped2, color="darkblue")
        axes[1, 1].axhline(0, color="red", linestyle="--")
        axes[1, 1].set_ylabel("Difference (AE - PCA) [%]")
    else:
        sns.barplot(ax=axes[1, 1], x=group_col2, y="Perc_Error_Change", data=grouped2, color="darkblue")
        axes[1, 1].axhline(grouped2["Perc_Error_Change"].mean(), color="red", linestyle="--")
        axes[1, 1].set_ylabel("(AE - PCA)/PCA [%]")
    axes[1, 1].set_xlabel(xlabel2)
    axes[1, 1].tick_params(axis='x', rotation=0)

    if wrapped_labels:
        axes[0, 0].set_xticklabels([fill(label, width=21) for label in grouped1[group_col1]])
        axes[1, 0].set_xticklabels([fill(label, width=21) for label in grouped1[group_col1]])
        axes[0, 1].set_xticklabels([fill(label, width=21) for label in grouped2[group_col2]])
        axes[1, 1].set_xticklabels([fill(label, width=21) for label in grouped2[group_col2]])
    
    plt.tight_layout()
    return fig, axes

## 4. Produce Figures (IMD & Density, OAC Supergroup)

In [ ]:
# Color palette
palette = {
    "AE": "seagreen",
    "PCA": "sandybrown"
}

print(f"Generating plots using median-error run (Run {median_run_idx}, RMSE = {rmse_per_run[median_run_idx]*100:.4f}%)")
print("#" * 50)

# Plot IMD and Density
fig, axes = plot_side_by_side_charts(
    reco_err, 
    pca_reco_err, 
    "IMD", 
    "Density_decile", 
    "IMD Decile", 
    "Density Decile", 
    "Comparison of Reconstruction Error", 
    palette,
    bottleneck=bottleneck,
    rotation=0,
    print_means=True
)
fig.suptitle(f'Reconstruction Error by Geography ({bottleneck}D, Median Run)', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(f"../plots/median_run_comparison_IMD_density_bottleneck_{bottleneck}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot OAC Supergroup and Group
fig, axes = plot_side_by_side_charts(
    reco_err, 
    pca_reco_err, 
    "Supergroup_codename", 
    "Group", 
    "OAC SuperGroup", 
    "OAC Group", 
    "Comparison of Reconstruction Error", 
    palette,
    bottleneck=bottleneck, 
    rotation=60,
    wrapped_labels=True
)
fig.suptitle(f'Reconstruction Error by OAC ({bottleneck}D, Median Run)', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(f"../plots/median_run_comparison_OAC_bottleneck_{bottleneck}.png", dpi=300, bbox_inches='tight')
plt.show()

## 5. Generate Spatial Data for Mapping

In [ ]:
"""
This section compares reconstruction errors between AE (median run) and PCA at two geographic levels:
  1. Output Area (OA21CD)
  2. Middle Layer Super Output Area (MSOA21CD)

Steps:
- Merge and align error data
- Attach geographic boundaries (OAs and MSOAs)
- Calculate % difference in error
- Export the results to Parquet files for mapping
"""

# STEP 1: Merge reconstruction errors at OA level
merged = reco_err.merge(pca_reco_err, on="OA21CD", suffixes=("", "_pca"))

# Remove duplicated columns from pca_reco_err unless they're actual PCA error columns
merged = merged[[col for col in merged.columns if not col.endswith("_pca") or col.startswith("pca_err")]]

# STEP 2: Load OA boundaries and merge with error data
oa_gdf = gpd.read_parquet("../data/geofiles/oabounds_2021approx.parquet")[["OA", "geometry"]]
oa_gdf = oa_gdf.rename(columns={"OA": "OA21CD"})  # match OA code field name
merged = oa_gdf.merge(merged, on="OA21CD", how="right")  # keep only OAs with error data

# STEP 3: Calculate % difference in error
merged[f"perc_diff_{bottleneck}"] = (
    (merged[f"reco_err_{bottleneck}"] - merged[f"pca_err_{bottleneck}"]) / merged[f"pca_err_{bottleneck}"] * 100
)

# STEP 4: Format and export OA-level results
merged.set_index("OA21CD", inplace=True)
merged = merged.round(5)
merged.to_parquet(f"../data/plots/maps/median_run_error_diff_by_OA_{bottleneck}d.parquet")
print(f"Saved OA-level results to ../data/plots/maps/median_run_error_diff_by_OA_{bottleneck}d.parquet")

In [ ]:
# STEP 5: Aggregate to MSOA level
# Keep only error columns and MSOA codes for aggregation
merged_for_msoa = merged[[col for col in merged.columns if col.startswith("reco_err") or col.startswith("pca_err") or col == "MSOA21CD"]]

# Group by MSOA and compute mean error values
msoa = merged_for_msoa.groupby("MSOA21CD").mean().reset_index()

# Recompute % difference in error at MSOA level
msoa[f"perc_diff_{bottleneck}"] = (
    (msoa[f"reco_err_{bottleneck}"] - msoa[f"pca_err_{bottleneck}"]) / msoa[f"pca_err_{bottleneck}"] * 100
)

# STEP 6: Load MSOA boundaries and merge with error data
msoa_gdf = gpd.read_file(
    "../data/geofiles/Middle_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFE_(V8)_and_RUC"
)[["MSOA21CD", "geometry"]]
msoa = msoa_gdf.merge(msoa, on="MSOA21CD", how="right")

# STEP 7: Export MSOA-level GeoDataFrame
msoa.to_parquet(f"../data/plots/maps/median_run_error_diff_by_MSOA_{bottleneck}d.parquet")
print(f"Saved MSOA-level results to ../data/plots/maps/median_run_error_diff_by_MSOA_{bottleneck}d.parquet")

In [ ]:
# Calculate quantiles for negative values in MSOA perc_diff to make figure 7

# Filter for negative values only
neg_values = msoa[msoa[f"perc_diff_{bottleneck}"] < 0][f"perc_diff_{bottleneck}"]
print(f"Only {len(msoa)-len(neg_values)} out of {len(msoa)} MSOAs have higher AE reconstruction error than PCA")

# Calculate quantile breaks
quantiles = neg_values.quantile([0, 0.2, 0.4, 0.6, 0.8, 1])
print(f"\nQuantiles for negative perc_diff_{bottleneck} values:\n{quantiles}")

## 6. Summary

In [ ]:
print("=" * 80)
print("SUMMARY: Error by Geography (Median-Error Run)")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Bottleneck dimension: {bottleneck}D")
print(f"  Total retraining runs available: {n_runs}")
print(f"  Selected run: Run {median_run_idx} (median overall RMSE)")
print(f"  Selected run RMSE: {rmse_per_run[median_run_idx]*100:.4f}%")
print(f"  Mean RMSE across all runs: {rmse_per_run.mean()*100:.4f}% +/- {rmse_per_run.std()*100:.4f}%")

print(f"\nReconstruction Error Comparison:")
print(f"  AE (median run) Mean RMSE: {reco_err[f'reco_err_{bottleneck}'].mean():.4f}%")
print(f"  PCA Mean RMSE: {pca_reco_err[f'pca_err_{bottleneck}'].mean():.4f}%")
print(f"  Improvement: {(pca_reco_err[f'pca_err_{bottleneck}'].mean() - reco_err[f'reco_err_{bottleneck}'].mean()) / pca_reco_err[f'pca_err_{bottleneck}'].mean() * 100:.2f}%")

print(f"\nOutput Files:")
print(f"  Plots: ../plots/median_run_comparison_*.png")
print(f"  Spatial data: ../data/plots/maps/median_run_error_diff_*.parquet")

print("\nRationale for using median-error run:")
print("  - Provides a single representative model result")
print("  - Avoids outliers (best/worst performing runs)")
print("  - More robust than arbitrary single run selection")
print("=" * 80)